In [2]:
import os
import subprocess
import sys
import re
import math

## visualize

In [3]:
import re

def parse_dat_file(dat_filepath):
    """
    .dat ファイルから BPM と availableTime をパースします。
    """
    bpm = 120 # デフォルト
    available_time_ms = []
    try:
        with open(dat_filepath, 'r') as f:
            content = f.read()

        # BPM抽出
        bpm_match = re.search(r'// BPM of the MIDI file: (\d+)', content)
        if bpm_match:
            bpm = int(bpm_match.group(1))
            print(f"INFO: BPM {bpm} をファイルから読み込みました。")
        else:
            print(f"警告: {dat_filepath} から BPM が見つかりません。デフォルトの {bpm} を使用します。")

        # availableTime抽出
        time_match = re.search(r'availableTime = \[(.*?)\];', content, re.DOTALL)
        if time_match:
            time_str = time_match.group(1).strip()
            available_time_ms = [int(t.strip()) for t in time_str.split(',') if t.strip()]
            print(f"INFO: availableTime を {len(available_time_ms)} 件読み込みました。")

        if not available_time_ms:
            print(f"警告: {dat_filepath} から 'availableTime' が見つかりませんでした。")

    except FileNotFoundError:
        print(f"エラー: {dat_filepath} が見つかりません。")
        return None, None
    except Exception as e:
        print(f"エラー: {dat_filepath} の読み込み中にエラーが発生しました: {e}")
        return None, None

    return bpm, available_time_ms

def parse_solution_file(solution_filepath):
    """
    CPLEXの出力ファイルから運指データをパースします。
    """
    notes_to_play = []
    try:
        with open(solution_filepath, 'r') as f:
            lines = f.readlines()
        
        header_found = False
        for line in lines:
            line = line.strip()
            if not line: continue

            if re.match(r'Note\s+String\s+Fret\s+Handposition\s+Finger', line):
                header_found = True
                continue

            if not header_found: continue
            if 'Objective:' in line or 'Cost Breakdown' in line or 'Total Cost' in line: break
            if line.startswith('---'): break 

            parts = re.split(r'\s+', line)
            
            if len(parts) >= 5:
                note_name = parts[0]
                
                if note_name == 'R':
                    notes_to_play.append({
                        'note': 'R',
                        'string': 0, 
                        'fret': 0, 
                        'finger': 'x'
                    })
                    continue

                try:
                    finger_val = parts[4]
                    parsed_finger = 'x' if finger_val.lower() == 'x' else int(finger_val)

                    notes_to_play.append({
                        'note': note_name,
                        'string': int(parts[1]),
                        'fret': int(parts[2]),
                        'finger': parsed_finger
                    })
                except (ValueError, IndexError):
                    pass

        if not notes_to_play:
             print(f"警告: {solution_filepath} からデータを抽出できませんでした。")

    except FileNotFoundError:
        print(f"エラー: {solution_filepath} が見つかりません。")
        return []
    except Exception as e:
        print(f"エラー: {solution_filepath} の読み込み中にエラー: {e}")
        return []
        
    return notes_to_play

def generate_rhythmic_tab(bpm, available_time_ms, notes_to_play):
    """
    パースされたデータからテキストタブ譜を生成します。
    ★変更点: スライド（同弦・同指移動）の検出と 's' 表記の追加
    """
    print("\n\n" + "="*80)
    print("                      タブ譜 (Tablature)")
    print("="*80)

    quarter_note_ms = 60000.0 / bpm
    rhythm_definitions = {
        "q.": (1.5, 14), "q": (1.0, 10), "e.": (0.75, 8),
        "et": (2.0/3.0, 8), "e": (0.5, 6), "s.": (0.375, 5),
        "st": (1.0/3.0, 4), "s": (0.25, 4), "sst": (1.0/6.0, 3),
        "t": (0.125, 3) 
    }
    bpm_scaled_rhythms = sorted(
        [(quarter_note_ms * info[0], (symbol, info[1])) for symbol, info in rhythm_definitions.items()],
        key=lambda x: x[0]
    )

    def get_rhythm_symbol(ms_value):
        if ms_value >= 900: return "START", 6
        closest_match = min(bpm_scaled_rhythms, key=lambda x: abs(x[0] - ms_value))
        ideal_ms, (symbol, width) = closest_match
        if abs(ideal_ms - ms_value) > (ideal_ms * 0.30) and ideal_ms > 0:
             return "?", 4
        return symbol, width

    print(f"--- TAbLATURE (BPM: {bpm}) ---\")")
    print("Legend: sN = Slide to fret N")
    print("-" * 84 + "\n")

    tab_lines = {
        'rhythm': [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: [], 'finger': []
    }
    line_width = 75
    current_line_length = 0
    
    num_notes = min(len(notes_to_play), len(available_time_ms))
    if len(notes_to_play) != len(available_time_ms):
        print(f"警告: データ数不一致 (運指:{len(notes_to_play)}, 時間:{len(available_time_ms)}) -> {num_notes}個のみ表示")

    labels = { 'rhythm': "Rhythm: ", 1: "e|", 2: "B|", 3: "G|", 4: "D|", 5: "A|", 6: "E|", 'finger': "Finger: " }
    label_width = max(len(l) for l in labels.values())
    
    for key in tab_lines:
        tab_lines[key].append(labels[key].rjust(label_width))
    current_line_length = label_width

    for i in range(num_notes):
        note_info = notes_to_play[i]
        ms_val = available_time_ms[i]
        
        is_rest = (note_info['note'] == 'R')
        
        # --- ★スライド判定ロジック ---
        is_slide = False
        if i > 0 and not is_rest:
            prev_info = notes_to_play[i-1]
            # 前の音が休符でなく、かつ弦が同じで、かつ指が同じ場合
            if prev_info['note'] != 'R':
                if prev_info['string'] == note_info['string']:
                    if prev_info['finger'] == note_info['finger']:
                         # さらに「フレットが異なる」場合のみスライド表記するなど調整も可能ですが、
                         # モデル定義上PCコストが発生する移動はすべてスライド割引対象なので
                         # 同一指・同一弦ならスライド扱いとします。
                         is_slide = True
        # ---------------------------

        rhythm_symbol, padding_width = get_rhythm_symbol(ms_val)
        
        s_idx = note_info['string']
        
        # フレット表記の生成
        if is_slide:
            f_str = "s" + str(note_info['fret']) # スライドなら 's7' のように表記
        else:
            f_str = str(note_info['fret'])
            
        h_str = str(note_info['finger'])
        
        if is_rest:
            f_str = ""
            h_str = ""
            if rhythm_symbol != "START":
                rhythm_symbol += "(R)"

        col_width = max(len(rhythm_symbol), len(f_str), len(h_str), padding_width - 1) + 1

        tab_lines['rhythm'].append(rhythm_symbol.ljust(col_width, '-'))
        tab_lines['finger'].append(h_str.ljust(col_width, '-'))

        for string_index in range(1, 7):
            if not is_rest and string_index == s_idx:
                tab_lines[string_index].append(f_str.ljust(col_width, '-'))
            else:
                tab_lines[string_index].append("-" * col_width)

        current_line_length += col_width

        if current_line_length > line_width:
            for key in ['rhythm', 1, 2, 3, 4, 5, 6, 'finger']:
                print("".join(tab_lines[key]))
            print("\n" + ("-" * (line_width + 2)) + "\n")

            tab_lines = { k: [] for k in tab_lines }
            for key in tab_lines:
                tab_lines[key].append(labels[key].rjust(label_width))
            current_line_length = label_width

    if current_line_length > label_width:
        for key in ['rhythm', 1, 2, 3, 4, 5, 6, 'finger']:
            print("".join(tab_lines[key]))

## 実行セル

In [ ]:
def main():
    # --- 1. ディレクトリとパスの設定 ---
    oplrun_executable = "/mnt/nfs/workspace/2024/Fumiaki_Unno/Bachelor/ILOG CPLEX/opl/bin/x86-64_linux/oplrun"

    # ディレクトリ名を定義
    models_dir = "models"
    datas_dir = "datas"
    results_dir = "results"

    # data_basename = input(f"Please insert the .dat file name (must be in '{datas_dir}' folder) >> ")
    data_basename = "complex_compare.dat"
    # --- (MODIFIED) 実行するファイル名を変更 ---
    model_basename = "modified_model.mod"

    # --- 3. 完全なファイルパスの構築 ---
    model_file = os.path.join(models_dir, model_basename)
    data_file = os.path.join(datas_dir, data_basename)

    # --- 4. 動的な出力ファイル名の生成 (results/ ディレクトリ内) ---

    # .dat と .mod の拡張子を除いた名前を取得
    data_name_no_ext = os.path.splitext(data_basename)[0]   
    model_name_no_ext = os.path.splitext(model_basename)[0] 

    # .dat名 と .mod名 を組み合わせてベース名を作成
    base_name = f"{data_name_no_ext}_{model_name_no_ext}"
    output_ext = ".txt"

    # 初期の出力ファイルパスを構築 
    output_filename = os.path.join(results_dir, f"{base_name}{output_ext}")
    counter = 1

    # 重複回避のロジック
    while os.path.exists(output_filename):
        output_filename = os.path.join(results_dir, f"{base_name} ({counter}){output_ext}")
        counter += 1

    print("="*80)
    print("                   OPL 実行ログ (CPLEX) - MM-based Model")
    print("="*80)
    print(f"モデル: {model_file}")
    print(f"データ: {data_file}")
    print(f"出力ファイル: {output_filename}\n")

    # --- 5. コマンドの実行とリアルタイム出力 ---
    command = [oplrun_executable, model_file, data_file]
    opl_success = False

    try:
        # OPL実行エンジンが見つかるか確認
        if not os.path.exists(oplrun_executable):
            raise FileNotFoundError(f"OPL実行エンジンが見つかりません: {oplrun_executable}")
            
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace', # エンコーディングエラーを置換
            bufsize=1
        )

        # 出力ファイルを開く
        with open(output_filename, 'w', encoding='utf-8') as f:
            for line in process.stdout:
                sys.stdout.write(line) # 画面にCPLEXのログを表示
                sys.stdout.flush()
                f.write(line) # ファイルに保存

        process.wait()

        if process.returncode == 0:
            print(f"\n--- OPL実行が正常に完了しました ---")
            opl_success = True
        else:
            print(f"\n--- エラー: OPLプロセスが失敗しました (終了コード: {process.returncode}) ---")

    except FileNotFoundError as e:
        print(f"\n--- エラー: OPL実行エンジンが見つかりません ---")
        print(f"パスを確認してください: {e}")
    except Exception as e:
        print(f"\n--- 予期せぬエラーが発生しました ---")
        print(e)

    # --- 6. タブ譜の生成 (OPL実行が成功した場合のみ) ---
    if opl_success:
        # data_file と output_filename (どちらも完全パス) を渡す
        print(f"\n'{data_file}' からリズム情報を読み込み中...")
        bpm, rhythms = parse_dat_file(data_file) 

        print(f"'{output_filename}' から運指データを読み込み中...")
        fingerings = parse_solution_file(output_filename) 

        if bpm and rhythms and fingerings:
            generate_rhythmic_tab(bpm, rhythms, fingerings)
        elif not bpm or not rhythms:
            print(f"\nエラー: {data_file} のパースに失敗しました。タブ譜は生成できません。")
        elif not fingerings:
            print(f"\nエラー: {output_filename} のパースに失敗しました。タブ譜は生成できません。")
        else:
            print("\nエラー: ファイルの読み込みに失敗したため、タブ譜を生成できませんでした。")
    else:
        
        print("\nOPLの実行に失敗したため、タブ譜は生成されませんでした。")

# 
if __name__ == "__main__":
    # 必要なディレクトリが存在しない場合は作成
    os.makedirs("models", exist_ok=True)
    os.makedirs("datas", exist_ok=True)
    os.makedirs("results", exist_ok=True)
   
    main()
    
    print("--- Pythonスクリプト実行終 ---")

                   OPL 実行ログ (CPLEX) - MM-based Model
モデル: models/modified_model.mod
データ: datas/complex_compare.dat
出力ファイル: results/complex_compare_modified_model (8).txt


<<< setup

INFO: Neck Thickness Mode = 3 -> Max Reduction = 0.3
INFO: Neck Factor Curve Exponent k = 8.00392278
INFO: Calculating v17 Stretch Costs...
INFO: Final Stretch Costs & Angles Calculated.
INFO: Time Logic Initialized.
  - Relax Threshold: 250 ms
INFO: Time Penalty Factors Pre-calculated.
Version identifier: 22.1.1.0 | 2022-11-28 | 9160aff4d
CPXPARAM_TimeLimit                               10000
Tried aggregator 2 times.
MIP Presolve eliminated 167493 rows and 977251 columns.
MIP Presolve modified 17428 coefficients.
Aggregator did 312 substitutions.
Reduced MIP has 4955 rows, 6113 columns, and 76860 nonzeros.
Reduced MIP has 5307 binaries, 0 generals, 0 SOSs, and 612 indicators.
Presolve time = 1.94 sec. (2293.89 ticks)
Probing fixed 543 vars, tightened 457 bounds.
Probing changed sense of 1189 constraints.